# Gold layer

Three business-facing tables, fully recomputed from Silver on every run instead
of merged. Correctly merging a partial update into a running aggregate is easy
to get wrong, and a full recompute is cheap enough at this data volume that
there's no reason to take on that risk. Run after `silver.py`.

Cancelled and refunded orders are excluded from every table here, applied once
through a shared filter. A cancelled or refunded order never became real
revenue, and filtering it in one place keeps every table consistent instead of
each one making its own call.

In [0]:
from pyspark.sql import functions as F

CATALOG = "ecommerce_project"

valid_orders = (
    spark.table(f"{CATALOG}.silver.orders")
    .filter(~F.col("status").isin("cancelled", "refunded"))
)

## Daily revenue

Grain: one row per `order_date`. Uses `orders.total_amount` directly instead
of re-summing `order_items`, since `total_amount` is already the derived,
validated total for each order (`compute_order_totals`, in
`data_generator/generate_order_items.py`), and this table doesn't need line-item detail.

In [0]:
daily_revenue = (
    valid_orders
    .groupBy("order_date")
    .agg(
        F.sum("total_amount").alias("revenue"),
        F.count("order_id").alias("order_count"),
    )
    .orderBy("order_date")
)

daily_revenue.write.format("delta").mode("overwrite").saveAsTable(f"{CATALOG}.gold.daily_revenue")
print(f"daily_revenue: {daily_revenue.count()} rows")

## Category revenue

Grain: one row per product `category`. This table needs line-item detail,
since category lives on `products`, not `orders`. It joins `order_items` to
`products` and computes revenue as `quantity * unit_price_at_purchase`, the
price captured at purchase time, not today's `products.price`.

In [0]:
order_items = spark.table(f"{CATALOG}.silver.order_items")
products = spark.table(f"{CATALOG}.silver.products")

category_revenue = (
    order_items
    .join(valid_orders.select("order_id"), "order_id")
    .join(products.select("product_id", "category"), "product_id")
    .withColumn("line_revenue", F.col("quantity") * F.col("unit_price_at_purchase"))
    .groupBy("category")
    .agg(
        F.sum("line_revenue").alias("revenue"),
        F.sum("quantity").alias("items_sold"),
    )
    .orderBy(F.col("revenue").desc())
)

category_revenue.write.format("delta").mode("overwrite").saveAsTable(f"{CATALOG}.gold.category_revenue")
print(f"category_revenue: {category_revenue.count()} rows")

## Customer lifetime value

Grain: one row per `customer_id`.

In [0]:
customer_lifetime_value = (
    valid_orders
    .groupBy("customer_id")
    .agg(
        F.sum("total_amount").alias("lifetime_revenue"),
        F.count("order_id").alias("order_count"),
        F.min("order_date").alias("first_order_date"),
        F.max("order_date").alias("last_order_date"),
    )
)

customer_lifetime_value.write.format("delta").mode("overwrite").saveAsTable(f"{CATALOG}.gold.customer_lifetime_value")
print(f"customer_lifetime_value: {customer_lifetime_value.count()} rows")